# 04 – Matching: generic mode vs local-aware mode

**Step 2 of the implementation plan.** One matcher, two modes.

| Component | Generic mode | Local-aware mode |
|---|---|---|
| **Semantic similarity** – sentence-transformer (`all-MiniLM-L6-v2`), candidate profile vs job advert | ✓ | ✓ (same) |
| **Skill overlap** – Jaccard similarity of skill sets | ESCO skills only | ESCO **+ local vocabulary** |
| **Credential fit** – share of the job's mandatory requirement groups the candidate meets, read from the candidate's *text*, with equivalence rules | – | ✓ |
| **Location** – jobs outside the candidate's reachable districts are down-weighted | – | ✓ |

```
base  = w × semantic + (1 − w) × skill_overlap
generic score = base
local score   = base × (0.5 + 0.5 × credential_fit) × (1 if reachable else LOCATION_PENALTY)
```

The weight `w` is tuned **separately for each mode** on a 20% validation set of candidates, so both get the same tuning effort. The other 80% (the test set) are evaluated in notebook 05.

Note: the local matcher uses *all* equivalence rules (verified and unverified) and reads credentials from the CV text, while the ground truth uses only verified rules and the true credentials. So the matcher can be wrong, which keeps the comparison honest.

In [ ]:
from pathlib import Path
import sys, json, hashlib, time
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.limpopo import (read_csv_any_encoding, LocalVocab, ndcg_at_k, reachable_districts)

LOCAL, PROCESSED = ROOT / "data" / "raw" / "local", ROOT / "data" / "processed"

jobs = pd.read_csv(PROCESSED / "jobs_clean.csv", dtype=str).fillna("")
jobreq = pd.read_csv(PROCESSED / "job_requirements.csv", dtype=str).fillna("")
jobs = jobs.merge(jobreq[["job_id", "family", "req_groups"]], on="job_id", how="left")
jobs["req_groups"] = jobs["req_groups"].map(json.loads)
cands = pd.read_csv(PROCESSED / "candidates.csv", dtype=str).fillna("")
gt = pd.read_csv(PROCESSED / "ground_truth.csv", dtype={"rel": int, "rel_occupation": int})
vocab_generic = pd.read_csv(PROCESSED / "vocab_generic.csv", dtype=str).fillna("")
lv = LocalVocab(read_csv_any_encoding(LOCAL / "local_vocabulary.csv"), read_csv_any_encoding(LOCAL / "equivalences.csv"))

JOB_IDX = {j: i for i, j in enumerate(jobs["job_id"])}
CAND_IDX = {c: i for i, c in enumerate(cands["candidate_id"])}
print(f"{len(cands)} candidates × {len(jobs)} jobs = {len(cands)*len(jobs):,} pairs")

## Configuration

In [ ]:
CONFIG = {
    "MODEL_NAME": "all-MiniLM-L6-v2",
    "LOCATION_PENALTY": 0.5,          # local mode: score multiplier for unreachable jobs
    "CREDENTIAL_FLOOR": 0.5,          # local mode: score multiplier when no requirement is met
    "W_GRID": [0.0, 0.25, 0.5, 0.75, 1.0],
    "VALIDATION_SHARE": 0.2,
    "TOP_K_SAVE": 20,
    "SEED": 42,
    "MIN_LABEL_CHARS": 4,             # ignore very short ESCO labels (noise)
}

## 1. Skill extraction from job adverts (ESCO, spaCy PhraseMatcher)

Every ESCO label (preferred and alternative) becomes a phrase pattern. Matching ignores case. Candidates already have ESCO skill IDs from notebook 03.

In [ ]:
import spacy
from spacy.matcher import PhraseMatcher

nlp = spacy.blank("en")
matcher = PhraseMatcher(nlp.vocab, attr="LOWER")
labels = vocab_generic[vocab_generic["label"].str.len() >= CONFIG["MIN_LABEL_CHARS"]]
label_to_skill = {}
for skill_id, grp in labels.groupby("skill_id"):
    docs = list(nlp.tokenizer.pipe(grp["label"].tolist()))
    matcher.add(skill_id, docs)
print(f"PhraseMatcher built from {len(labels):,} ESCO labels")

def esco_skills(text):
    doc = nlp.make_doc(text)
    return {nlp.vocab.strings[mid] for mid, s, e in matcher(doc)}

t0 = time.time()
jobs["esco_skills"] = jobs["description"].map(esco_skills)
print(f"Extracted in {time.time()-t0:.1f}s; ESCO skills per job: mean {jobs['esco_skills'].map(len).mean():.1f}, "
      f"jobs with none: {(jobs['esco_skills'].map(len) == 0).mean():.0%}")

## 2. Skill and credential sets for both sides

In [ ]:
# Candidates: ESCO skills from notebook 03 (structured), local items read from their CV text
cands["esco_skills"] = cands["skill_uris"].map(lambda s: set(s.split(";")) if s else set())
cands["local_items"] = cands["profile_text"].map(lv.find)
jobs["local_items"] = jobs["description"].map(lv.find)

# Generic sets: ESCO only. Local sets: ESCO + local vocabulary ids.
C_GEN = cands["esco_skills"].tolist()
C_LOC = [a | {"local:" + x for x in b} for a, b in zip(cands["esco_skills"], cands["local_items"])]
J_GEN = jobs["esco_skills"].tolist()
J_LOC = [a | {"local:" + x for x in b} for a, b in zip(jobs["esco_skills"], jobs["local_items"])]

def jaccard_matrix(A, B):
    M = np.zeros((len(A), len(B)), dtype=np.float32)
    for i, a in enumerate(A):
        if not a: continue
        for j, b in enumerate(B):
            if b:
                inter = len(a & b)
                if inter: M[i, j] = inter / len(a | b)
    return M

overlap_gen = jaccard_matrix(C_GEN, J_GEN)
overlap_loc = jaccard_matrix(C_LOC, J_LOC)
print(f"Pairs with any overlap – generic: {(overlap_gen > 0).mean():.1%}, local: {(overlap_loc > 0).mean():.1%}")

## 3. Credential fit and location (local-aware mode only)

`credential_fit` = share of the job's mandatory requirement groups met by the credentials found in the candidate's **text**, after applying equivalence rules (all rules, including unverified). Jobs with no listed requirement get 1.

In [ ]:
cred_closure = [lv.closure(items, verified_only=False) for items in cands["local_items"]]
cred_fit = np.ones((len(cands), len(jobs)), dtype=np.float32)
for j, groups in enumerate(jobs["req_groups"]):
    if not groups: continue
    for i, met in enumerate(cred_closure):
        cred_fit[i, j] = sum(any(g in met for g in grp) for grp in groups) / len(groups)

reach = [set(r.split(";")) for r in cands["reachable_districts"]]
reachable = np.array([[d in r for d in jobs["district"]] for r in reach], dtype=bool)
print(f"Average credential fit: {cred_fit.mean():.2f}; share of pairs reachable: {reachable.mean():.0%}")

## 4. Semantic similarity (sentence-transformers)

The model (~90 MB) downloads the first time. Embeddings are cached in `data/processed/`, so re-runs are fast. If the model cannot be loaded (e.g. no internet), the notebook falls back to TF-IDF cosine similarity and says so; **report which one you used**.

In [ ]:
job_texts = (jobs["title"] + ". " + jobs["description"]).tolist()
cand_texts = cands["profile_text"].tolist()
key = hashlib.md5(("".join(job_texts) + "".join(cand_texts) + CONFIG["MODEL_NAME"]).encode()).hexdigest()[:10]
cache = PROCESSED / f"embeddings_{key}.npz"

SEMANTIC_METHOD = None
if cache.exists():
    z = np.load(cache); Ej, Ec = z["jobs"], z["cands"]; SEMANTIC_METHOD = str(z["method"])
    print("Loaded cached embeddings:", cache.name)
else:
    try:
        from sentence_transformers import SentenceTransformer
        model = SentenceTransformer(CONFIG["MODEL_NAME"])
        Ej = model.encode(job_texts, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
        Ec = model.encode(cand_texts, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
        SEMANTIC_METHOD = f"sentence-transformers/{CONFIG['MODEL_NAME']}"
    except Exception as e:
        print("Sentence-transformer unavailable -> TF-IDF fallback. Reason:", type(e).__name__, str(e)[:150])
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.preprocessing import normalize
        vec = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=1, sublinear_tf=True)
        X = vec.fit_transform(job_texts + cand_texts)
        X = normalize(X)
        Ej, Ec = X[:len(job_texts)].toarray(), X[len(job_texts):].toarray()
        SEMANTIC_METHOD = "tfidf-fallback"
    np.savez_compressed(cache, jobs=Ej, cands=Ec, method=SEMANTIC_METHOD)

semantic = np.clip(Ec @ Ej.T, 0, 1).astype(np.float32)
print("Semantic method:", SEMANTIC_METHOD, "| mean similarity:", round(float(semantic.mean()), 3))

## 5. Ground-truth matrix and validation/test split

In [ ]:
REL = np.zeros((len(cands), len(jobs)), dtype=np.int8)
REL_OCC = np.zeros_like(REL)
ci = gt["candidate_id"].map(CAND_IDX).to_numpy(); ji = gt["job_id"].map(JOB_IDX).to_numpy()
REL[ci, ji] = gt["rel"].to_numpy(); REL_OCC[ci, ji] = gt["rel_occupation"].to_numpy()

rng = np.random.default_rng(CONFIG["SEED"])
split = pd.Series("test", index=cands["candidate_id"])
for scen, grp in cands.groupby("scenario"):                       # stratified by scenario
    ids = grp["candidate_id"].to_numpy()
    n_val = int(round(len(ids) * CONFIG["VALIDATION_SHARE"]))
    split[rng.choice(ids, size=n_val, replace=False)] = "validation"
cands["split"] = split.values
print(cands.groupby(["split", "scenario"]).size().unstack())

## 6. Scores and tuning `w` on the validation set

In [ ]:
def generic_scores(w):
    return w * semantic + (1 - w) * overlap_gen

def local_scores(w):
    base = w * semantic + (1 - w) * overlap_loc
    cred = CONFIG["CREDENTIAL_FLOOR"] + (1 - CONFIG["CREDENTIAL_FLOOR"]) * cred_fit
    loc = np.where(reachable, 1.0, CONFIG["LOCATION_PENALTY"])
    return base * cred * loc

def mean_ndcg(S, rows, rel=REL, k=10):
    vals = []
    for i in rows:
        order = np.argsort(-S[i], kind="stable")
        vals.append(ndcg_at_k(rel[i, order], rel[i], k))
    return float(np.nanmean(vals))

val_rows = np.where(cands["split"] == "validation")[0]
tuning = []
for w in CONFIG["W_GRID"]:
    tuning.append({"w": w, "generic_ndcg10": mean_ndcg(generic_scores(w), val_rows),
                   "local_ndcg10": mean_ndcg(local_scores(w), val_rows)})
tuning = pd.DataFrame(tuning)
BEST = {"generic": float(tuning.loc[tuning["generic_ndcg10"].idxmax(), "w"]),
        "local": float(tuning.loc[tuning["local_ndcg10"].idxmax(), "w"])}
print("Best w per mode (validation NDCG@10):", BEST)
tuning.round(3)

## 7. Final rankings

In [ ]:
S_GEN, S_LOC = generic_scores(BEST["generic"]), local_scores(BEST["local"])
K = CONFIG["TOP_K_SAVE"]
rows = []
for mode, S in [("generic", S_GEN), ("local", S_LOC)]:
    for i, cid in enumerate(cands["candidate_id"]):
        order = np.argsort(-S[i], kind="stable")[:K]
        for rank, j in enumerate(order, 1):
            rows.append({"candidate_id": cid, "mode": mode, "rank": rank, "job_id": jobs.at[j, "job_id"],
                         "score": float(S[i, j]), "semantic": float(semantic[i, j]),
                         "overlap": float((overlap_gen if mode == "generic" else overlap_loc)[i, j]),
                         "credential_fit": float(cred_fit[i, j]), "reachable": bool(reachable[i, j]),
                         "rel": int(REL[i, j]), "rel_occupation": int(REL_OCC[i, j])})
rankings = pd.DataFrame(rows)
print(f"Saved top {K} jobs per candidate per mode: {len(rankings):,} rows")

## 8. Worked example (for the write-up)

An *equivalent*-scenario candidate from the test set (CV in local wording, e.g. NC(V), Code 10), chosen as the one where the two modes differ most. This is an illustration, not evidence: the averages in notebook 05 are the evidence.

In [ ]:
TITLE = dict(zip(jobs["job_id"], jobs["title"])); DIST = dict(zip(jobs["job_id"], jobs["district"]))
# Pick the equivalent-scenario test candidate whose top-5 relevance differs most between the modes
top5 = rankings[rankings["rank"] <= 5].groupby(["candidate_id", "mode"])["rel"].sum().unstack()
pool = cands[(cands["split"] == "test") & (cands["scenario"] == "equivalent")]["candidate_id"]
diff = (top5.loc[pool, "local"] - top5.loc[pool, "generic"]).sort_values(ascending=False)
ex = cands[cands["candidate_id"] == diff.index[0]].iloc[0]
print(f"Example chosen because its top-5 relevance differs most between modes (local − generic = {diff.iloc[0]}).\n")
print(ex["profile_text"], "\n")
for mode in ["generic", "local"]:
    top = rankings[(rankings["candidate_id"] == ex["candidate_id"]) & (rankings["mode"] == mode)].head(5)
    print(f"--- {mode.upper()} MODE ---")
    for _, r in top.iterrows():
        print(f"{r['rank']}. {TITLE[r['job_id']][:60]:60s} | {DIST[r['job_id']]:10s} | fit {r['credential_fit']:.2f} | rel {r['rel']}")
    print()

## 9. Skills gap for each candidate's top 3 jobs (local-aware mode)

For each recommended job: the mandatory requirement groups the candidate does not meet, and the ESCO skills in the advert the candidate lacks. Notebook 05 maps these gaps to Limpopo training options.

In [ ]:
SKILL_NAME = dict(zip(vocab_generic["skill_id"], vocab_generic["skill_name"]))
JOB_ROW = {j: r for j, r in jobs.set_index("job_id").iterrows()}
gap_rows = []
for _, r in rankings[(rankings["mode"] == "local") & (rankings["rank"] <= 3)].iterrows():
    i = CAND_IDX[r["candidate_id"]]; job = JOB_ROW[r["job_id"]]
    met = cred_closure[i]
    missing_groups = [grp for grp in job["req_groups"] if not any(g in met for g in grp)]
    missing_skills = sorted(job["esco_skills"] - cands.at[i, "esco_skills"])
    gap_rows.append({"candidate_id": r["candidate_id"], "rank": r["rank"], "job_id": r["job_id"],
                     "missing_credentials": " | ".join(" or ".join(lv.label[g] for g in grp) for grp in missing_groups),
                     "missing_credential_ids": json.dumps(missing_groups),
                     "missing_skills": "; ".join(SKILL_NAME.get(s, s) for s in missing_skills[:8]),
                     "n_missing_skills": len(missing_skills)})
gaps = pd.DataFrame(gap_rows)
print(f"Top-3 recommendations with a credential gap: {(gaps['missing_credentials'] != '').mean():.0%}")
gaps[gaps["missing_credentials"] != ""].head(8)[["candidate_id", "rank", "job_id", "missing_credentials"]]

## 10. Save

In [ ]:
np.savez_compressed(PROCESSED / "match_scores.npz", generic=S_GEN, local=S_LOC, semantic=semantic,
                    overlap_generic=overlap_gen, overlap_local=overlap_loc, credential_fit=cred_fit,
                    reachable=reachable, rel=REL, rel_occupation=REL_OCC,
                    candidate_ids=cands["candidate_id"].to_numpy(), job_ids=jobs["job_id"].to_numpy())
rankings.to_csv(PROCESSED / "rankings.csv", index=False)
gaps.to_csv(PROCESSED / "skills_gap.csv", index=False)
cands[["candidate_id", "split"]].to_csv(PROCESSED / "candidate_split.csv", index=False)
tuning.to_csv(PROCESSED / "tuning_results.csv", index=False)
json.dump({**CONFIG, "best_w": BEST, "semantic_method": SEMANTIC_METHOD}, open(PROCESSED / "matcher_config.json", "w"), indent=2)
for f in ["match_scores.npz", "rankings.csv", "skills_gap.csv", "candidate_split.csv", "tuning_results.csv", "matcher_config.json"]:
    print(f"{f:24s} {(PROCESSED / f).stat().st_size/1024:8.0f} KB")

### Notes for the write-up
- **Semantic method:** state which was used (`matcher_config.json` → `semantic_method`). The intended method is `all-MiniLM-L6-v2`; TF-IDF is only a fallback.
- **Tuning:** `w` was tuned per mode on the validation set (20%, stratified by scenario); report the tuning table. The credential floor (0.5) and location penalty (0.5) were fixed, not tuned; mention them as design choices.
- **Skill extraction quality** from ESCO labels has not yet been measured. The plan calls for hand-labelling 50–100 adverts to report precision/recall/F1; do this if time allows, otherwise list it as a limitation.
- **Skill overlap is sparse** because job texts are summaries; most of the ranking signal comes from semantic similarity and, in local mode, credential fit and location.
- **Credentials are read from text** in local mode, using all equivalence rules (some unverified), so the matcher can make mistakes the ground truth does not.